<div style="
  background: linear-gradient(135deg, #ff9a9e, #fad0c4, #fbc2eb);
  padding: 28px;
  border-radius: 24px;
  text-align: center;
  color: #3a2c4a;
  box-shadow: 0 8px 20px rgba(0,0,0,0.12);
">
  <h1 style="font-size: 38px; margin-bottom: 8px;">☕ Café Queue Simulation</h1>
  <h2 style="font-size: 22px; margin-top: 0;">Discrete-Probability Modeling of Arrivals, Menu Choice & Service</h2>
  <p style="font-size: 17px;">Poisson arrivals, categorical menu selection, and geometric service times with a dynamic third server. Seed 9248 everywhere for full reproducibility.</p>
</div>

## Model at a glance

Time is slotted in 5-minute intervals indexed by  = 0, 1, 2, \dots$. In each slot $:

- Arrivals  \overset{\mathrm{iid}}{\sim} \mathrm{Poisson}(\lambda)$ with base $\lambda = 0.8$ per slot.
- Each arrival picks one of 7 menu items from a categorical distribution with probabilities $.
- Each customer needs a geometric number of service slots  \sim \mathrm{Geom}(p_i)$ on $\{1, 2, \dots\}$, with (T_i = k) = (1 - p_i)^{k-1} p_i$.
- Two servers are always active; a third server activates in slot $ when {t-1} - 2 \ge h$ with default threshold  = 4$.
- Each active server makes one Bernoulli($) attempt per slot on a head-of-line customer; success completes that service.

**Menu selection probabilities $ and per-slot service success probabilities $:**

| Item | $ | $ |
|---|---|---|
| Coffee | 0.25 | 0.40 |
| Cake | 0.15 | 0.35 |
| Smoothie | 0.15 | 0.30 |
| Shake | 0.10 | 0.35 |
| Sandwich | 0.10 | 0.25 |
| Tea | 0.15 | 0.35 |
| Ice Cream | 0.10 | 0.30 |


In [ ]:
SEED <- 9248
set.seed(SEED)

library(ggplot2)
library(dplyr)
library(tidyr)
library(gridExtra)
library(knitr)

lambda_base <- 0.8
base_servers <- 2L
h_default <- 4L
slot_minutes <- 5

menu_q <- c(Coffee = 0.25, Cake = 0.15, Smoothie = 0.15, Shake = 0.10,
            Sandwich = 0.10, Tea = 0.15, Ice_Cream = 0.10)
menu_p <- c(Coffee = 0.40, Cake = 0.35, Smoothie = 0.30, Shake = 0.35,
            Sandwich = 0.25, Tea = 0.35, Ice_Cream = 0.30)
menu_names <- names(menu_q)

dir.create("results", showWarnings = FALSE)
kable(data.frame(Item = menu_names, q = as.numeric(menu_q), p = as.numeric(menu_p)),
      caption = "Menu selection probabilities q and per-attempt success probabilities p")


<div style="
  background: linear-gradient(135deg, #74c69d, #48cae4);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">🚬 Warm-Up: Banach Matchbox</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">Markov chain view, corrected PMF, and simulation check</p>
</div>

### The chain

Two boxes start full: $S_0 = (n, n)$. After $t$ draws the state is $S_t = (x_1, x_2)$, the matches still left in each box. Each step picks box 1 or box 2 with probability $1/2$ and removes one match:

$$S_{t+1} = \begin{cases} (x_1 - 1, x_2) & \text{with prob. } 1/2, \\ (x_1, x_2 - 1) & \text{with prob. } 1/2. \end{cases}$$

Only the current pair matters for the next step, so $P(S_{t+1} \mid S_t, \dots, S_0) = P(S_{t+1} \mid S_t)$: a discrete-time Markov chain on the $(x_1, x_2)$ grid. The walk stops at the first hitting time $\tau = \min\{t: x_1 = 0 \text{ or } x_2 = 0\}$, and we record $K = \max(x_1, x_2)$ at time $\tau$.

### Corrected distribution of $K$

Suppose box 1 is the one that empties first with $K = k \ge 1$ left in box 2. Then box 1 was picked exactly $n$ times, box 2 exactly $n - k$ times, and the final pick must be box 1. The first $2n - k - 1$ picks can be arranged freely:

$$P(\text{box 1 empties first}, K = k) = \binom{2n - k - 1}{n - 1} \left(\tfrac{1}{2}\right)^{2n - k}.$$

By symmetry the same holds for box 2, so for $k = 1, \dots, n$:

$$P(K = k) = 2\binom{2n - k - 1}{n - 1} \left(\tfrac{1}{2}\right)^{2n - k} = \binom{2n - k - 1}{n - 1} \left(\tfrac{1}{2}\right)^{2n - k - 1}.$$

Crucially, **$P(K = 0) = 0$** under this stopping rule: reaching $(0, 0)$ needs $2n$ draws, but one box is already empty after at most $2n - 1$ draws, so the walk always stops earlier. A formula such as $\binom{2n-k}{n}/2^{2n-k}$ assigns $P(K=0) \approx 0.176$ for $n = 10$ and halves the $k = n$ mass; it belongs to a different variant (empty box *discovered on the next attempt*) and must not be used here.

### Simulation check

The code below evaluates the corrected PMF for $n = 10$, simulates $100{,}000$ walks with seed 9248, tabulates the empirical distribution, and overlays theory against simulation. Both the figure and the comparison table are saved under `results/`.


In [ ]:
set.seed(9248)

banach_pmf <- function(n) {
  k <- 1:n
  p <- 2 * choose(2 * n - k - 1, n - 1) / 2^(2 * n - k)
  data.frame(k = k, analytical = p)
}

simulate_banach_once <- function(n) {
  box1 <- n
  box2 <- n
  while (box1 > 0 && box2 > 0) {
    if (runif(1) < 0.5) box1 <- box1 - 1 else box2 <- box2 - 1
  }
  max(box1, box2)
}

n_banach <- 10L
N_banach <- 100000L
sim_draws <- replicate(N_banach, simulate_banach_once(n_banach))
sim_tab <- as.data.frame(table(factor(sim_draws, levels = 1:n_banach)))
colnames(sim_tab) <- c("k", "count")
sim_tab$k <- as.integer(as.character(sim_tab$k))
sim_tab$simulated <- sim_tab$count / sum(sim_tab$count)

banach_cmp <- merge(banach_pmf(n_banach), sim_tab[, c("k", "simulated")], by = "k")
print(banach_cmp, digits = 5, row.names = FALSE)
cat(sprintf("\nP(K = 0) is structurally 0 here; every walk stops with K >= 1 (min simulated K = %d).\n", min(sim_draws)))

p_banach <- ggplot(banach_cmp, aes(x = k)) +
  geom_col(aes(y = analytical), fill = "steelblue", alpha = 0.6) +
  geom_point(aes(y = simulated), color = "red", size = 2) +
  geom_line(aes(y = simulated), color = "red", linewidth = 0.8) +
  labs(title = "Banach matchbox (n = 10): corrected theory vs simulation",
       x = "Matches left in the other box (k)", y = "Probability") +
  theme_minimal()
print(p_banach)
ggsave("results/banach_comparison.png", p_banach, width = 7, height = 4.5, dpi = 150)
write.csv(banach_cmp, "results/banach_comparison.csv", row.names = FALSE)


<div style="
  background: linear-gradient(135deg, #e8b7e8, #ffb6f4);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">📦 Building Blocks</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">Poisson arrivals, categorical menu, geometric service times</p>
</div>

<div style="
  background: linear-gradient(135deg, #9668b1, #8fb5d7, #78b9f6);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">🎲 Menu-Aware Sampling</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">Empirical means, proportions, histograms, boxplots, ECDFs</p>
</div>

<div style="
  background: linear-gradient(135deg, #7dbcf3, #9ff98d, #d6a870);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">⏱️ Per-Item Service Analytics</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">Expected service times and bottleneck identification</p>
</div>

<div style="
  background: linear-gradient(135deg, #f197a7, #efb66f, #f9fc95);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">🏪 FIFO Café Queue Simulator</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">One FIFO discipline with dynamic third-server activation</p>
</div>

<div style="
  background: linear-gradient(135deg, #74c69d, #b7e4c7, #48cae4);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">📈 Arrival-Rate Sensitivity</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">How E[Q], Var(Q), P(Q=0), and third-server use evolve with λ</p>
</div>

<div style="
  background: linear-gradient(135deg, #cdb4db, #ffc8dd, #ffafcc);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">🎛️ Activation-Threshold Sensitivity</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">Sweeping h at fixed λ = 0.8</p>
</div>

<div style="
  background: linear-gradient(135deg, #90be6d, #43aa8b, #577590);
  padding: 20px 26px;
  border-radius: 20px;
  text-align: center;
  color: #1d2a3a;
  box-shadow: 0 6px 16px rgba(0,0,0,0.10);
">
  <h2 style="margin: 0; font-size: 28px;">🗺️ Joint λ–h Stability Map</h2>
  <p style="margin: 6px 0 0 0; font-size: 16px;">Where the café stays under control</p>
</div>

## Takeaways

(Filled in as each analysis section lands.)


In [ ]:
sessionInfo()
